In [0]:
CATALOG = "workspace"
DATABASE = "qualite_eau"
RAW_PATH = f"/Volumes/{CATALOG}/{DATABASE}/raw"
BRONZE_PATH = f"/Volumes/{CATALOG}/{DATABASE}/bronze"

In [0]:
from pyspark.sql.functions import current_timestamp, regexp_extract, col

df_raw = (
    spark.read
    .option("multiLine", "true")
    .json(f"{RAW_PATH}/**/*.json")
    .withColumn("_source_file", col("_metadata.file_path"))
    .withColumn("_ingestion_timestamp", current_timestamp())
    .withColumn("_departement", regexp_extract(col("_metadata.file_path"), r"dept_(\d+)", 1))
    .withColumn("_annee", regexp_extract(col("_metadata.file_path"), r"resultats_(\d+)", 1))
)

print(f"Lignes lues : {df_raw.count():,}")
print(f"Colonnes : {len(df_raw.columns)}")
df_raw.printSchema()

In [0]:
display(df_raw.limit(5))

In [0]:
(df_raw.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .partitionBy("_departement", "_annee")
    .saveAsTable(f"{CATALOG}.{DATABASE}.bronze_qualite_eau"))

print("Table Bronze écrite")

In [0]:
display(spark.sql(f"""
    SELECT _departement, _annee, COUNT(*) as nb_lignes
    FROM {CATALOG}.{DATABASE}.bronze_qualite_eau
    GROUP BY _departement, _annee
    ORDER BY _departement, _annee
"""))